<a href="https://colab.research.google.com/github/donald-ye/mitr-take-home-eval/blob/main/mitr_infonce_generalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part D: MITR-InfoNCE Generalization to BERT & RoBERTa

## Experiment 1 from Part C

**Hypothesis**: If we run MITR-InfoNCE on `bert-base-uncased` and `roberta-base` using identical hyperparameters to the existing Cosine/CKA experiments, then InfoNCE will improve both accuracy and contradiction rate on at least one backbone, because it is the only strategy that achieves this on DistilBERT and the contrastive objective should generalize to 12-layer models.

### Why this experiment matters
In `results.md`, MITR-InfoNCE is the **only strategy** that improves both accuracy (+0.20%) and contradiction rate (-1.80%) simultaneously on DistilBERT. Yet `roberta_bert_results.md` explicitly excludes it: *"InfoNCE skipped for scope."* This notebook fills that gap.

### Colab scope-down
- **Epochs**: 5
- **Precision**: fp16
- **Batch size**: 16 with grad_accum=4 (effective 64, matches original)
- **Data**: 8000 train / 1500 val (same as original)

Full experiment (5 epochs, A100, bf16)
> **Start training early. Work on Part E while it runs.**

In [ ]:
!pip install -q --upgrade "transformers>=4.36" datasets accelerate matplotlib seaborn tqdm
print("Done.")

In [ ]:
import os, json, random, warnings
from dataclasses import dataclass
from typing import Dict, List, Optional
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoModel, AutoTokenizer,
    get_cosine_schedule_with_warmup,
)
from datasets import load_dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

In [ ]:
@dataclass
class Config:
    model_name: str       = "bert-base-uncased"
    num_labels: int       = 2
    batch_size: int       = 16     # T4 safe; effective batch = 16*4 = 64 via grad accum
    eval_batch_size: int  = 32
    learning_rate: float  = 2e-5
    weight_decay: float   = 0.01
    num_epochs: int       = 5
    warmup_ratio: float   = 0.06
    max_grad_norm: float  = 1.0
    grad_accum_steps: int = 4      # effective batch = 64
    mi_lambda: float      = 0.01
    mi_warmup_steps: int  = 200
    club_hidden: int      = 384
    mi_strategy: str      = "infonce"
    dataset: str          = "boolq"
    max_length: int       = 256
    num_train: int        = 8000
    num_eval: int         = 1500
    num_contra: int       = 500
    num_workers: int      = 2
    use_fp16: bool        = True   # fp16 for T4 (bf16 preferred on A100)
    seed: int             = 42
    output_dir: str       = "experiment_results"

BACKBONES   = ["bert-base-uncased", "roberta-base"]
# Run baseline + InfoNCE only (this is the gap we are filling)
STRATEGIES  = ["baseline", "infonce"]

print("Config defined.")
print(f"Backbones : {BACKBONES}")
print(f"Strategies: {STRATEGIES}")

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark        = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Use fp16 on T4; bf16 on A100
if torch.cuda.is_available():
    if torch.cuda.is_bf16_supported():
        DTYPE = torch.bfloat16
        print("Using bf16 (A100 detected)")
    else:
        DTYPE = torch.float16
        print("Using fp16 (T4 detected)")
else:
    DTYPE = torch.float32

print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {props.name}")
    print(f"VRAM   : {props.total_memory / 1e9:.1f} GB")

os.makedirs("experiment_results", exist_ok=True)

In [ ]:
# -- Load BoolQ --
def load_boolq():
    ds = load_dataset("google/boolq")
    def fmt(ex):
        return {
            "text":     ex["question"] + " [SEP] " + ex["passage"][:400],
            "label":    int(ex["answer"]),
            "question": ex["question"],
        }
    train = ds["train"].map(fmt, remove_columns=ds["train"].column_names)
    val   = ds["validation"].map(fmt, remove_columns=ds["validation"].column_names)
    return train, val

rng = random.Random(42)
def subsample(dataset, n):
    if n < 0 or n >= len(dataset):
        return dataset
    return dataset.select(rng.sample(range(len(dataset)), n))

print("Loading BoolQ...")
train_raw, val_raw = load_boolq()
train_raw = subsample(train_raw, 8000)
val_raw   = subsample(val_raw,   1500)
print(f"Train: {len(train_raw):,}  Val: {len(val_raw):,}")

In [ ]:
# -- Contradiction pairs (same logic as original notebooks) --
_AUX = [
    ("is ", "is not "), ("are ", "are not "), ("was ", "was not "),
    ("were ", "were not "), ("does ", "does not "), ("do ", "do not "),
    ("did ", "did not "), ("has ", "has not "), ("have ", "have not "),
    ("had ", "had not "), ("can ", "cannot "), ("could ", "could not "),
    ("will ", "will not "), ("would ", "would not "), ("should ", "should not "),
]

def negate_question(q):
    q = q.strip().rstrip("?").lower()
    for pos, neg in _AUX:
        if q.startswith(neg):  return pos + q[len(neg):]
        if q.startswith(pos):  return neg + q[len(pos):]
    return None

def create_contradiction_pairs(dataset, n_pairs):
    pairs = []
    for ex in dataset:
        q = ex.get("question", "").strip()
        if not q: continue
        q_neg = negate_question(q)
        if q_neg is None or q_neg.strip() == q.strip(): continue
        text_fwd = ex["text"]
        prefix   = text_fwd.split("[SEP]")[0] if "[SEP]" in text_fwd else ""
        text_neg = prefix + "[SEP] " + q_neg
        pairs.append({
            "text_forward": text_fwd, "text_negated": text_neg,
            "label_forward": ex["label"], "label_negated": 1 - ex["label"],
            "question": q, "negated_question": q_neg,
        })
        if len(pairs) >= n_pairs: break
    return pairs

contradiction_pairs = create_contradiction_pairs(val_raw, 500)
print(f"Contradiction pairs: {len(contradiction_pairs):,}")

In [ ]:
class LogicDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        enc = tokenizer([ex["text"] for ex in data], max_length=max_length,
                        padding="max_length", truncation=True, return_tensors="pt")
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels         = torch.tensor([ex["label"] for ex in data], dtype=torch.long)
    def __len__(self):   return len(self.labels)
    def __getitem__(self, i):
        return {"input_ids": self.input_ids[i],
                "attention_mask": self.attention_mask[i],
                "labels": self.labels[i]}

class ContradictionPairDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length):
        fwd = tokenizer([p["text_forward"] for p in pairs], max_length=max_length,
                        padding="max_length", truncation=True, return_tensors="pt")
        neg = tokenizer([p["text_negated"]  for p in pairs], max_length=max_length,
                        padding="max_length", truncation=True, return_tensors="pt")
        self.fwd_ids  = fwd["input_ids"];   self.fwd_mask = fwd["attention_mask"]
        self.neg_ids  = neg["input_ids"];   self.neg_mask = neg["attention_mask"]
        self.fwd_lbl  = torch.tensor([p["label_forward"] for p in pairs], dtype=torch.long)
        self.neg_lbl  = torch.tensor([p["label_negated"]  for p in pairs], dtype=torch.long)
    def __len__(self):   return len(self.fwd_lbl)
    def __getitem__(self, i):
        return {"fwd_input_ids": self.fwd_ids[i], "fwd_attention_mask": self.fwd_mask[i],
                "neg_input_ids": self.neg_ids[i], "neg_attention_mask": self.neg_mask[i],
                "fwd_label": self.fwd_lbl[i],     "neg_label": self.neg_lbl[i]}

print("Dataset classes defined.")

In [ ]:
# -- InfoNCE MI Estimator --
# Contrastive lower bound on MI.
# This is the strategy missing from the original BERT/RoBERTa experiments.

class InfoNCEMI(nn.Module):
    def __init__(self, x_dim, y_dim, hidden, temperature=0.07):
        super().__init__()
        self.proj_x = nn.Sequential(
            nn.Linear(x_dim, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden),
        )
        self.proj_y = nn.Sequential(
            nn.Linear(y_dim, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden),
        )
        self.temperature = temperature

    def forward(self, x, y):
        x_proj = F.normalize(self.proj_x(x.float()), dim=-1)
        y_proj = F.normalize(self.proj_y(y.float()), dim=-1)
        sim    = torch.matmul(x_proj, y_proj.T) / self.temperature
        labels = torch.arange(x.size(0), device=x.device)
        loss   = F.cross_entropy(sim, labels)
        mi     = torch.log(torch.tensor(float(x.size(0)), device=x.device)) - loss
        return mi.clamp(-20.0, 20.0)

print("InfoNCEMI defined.")

In [ ]:
class BaselineClassifier(nn.Module):
    def __init__(self, model_name, num_labels=2):
        super().__init__()
        self.encoder        = AutoModel.from_pretrained(model_name)
        h                   = self.encoder.config.hidden_size
        self.pre_classifier = nn.Linear(h, h)
        self.classifier     = nn.Linear(h, num_labels)
        self.dropout        = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask, labels=None, is_training=False):
        out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls    = out.last_hidden_state[:, 0]
        cls    = self.dropout(F.relu(self.pre_classifier(cls)))
        logits = self.classifier(cls)
        result = {"logits": logits, "mi_loss": 0.0}
        if labels is not None:
            result["loss"] = F.cross_entropy(logits, labels)
        return result


class MITRInfoNCEClassifier(nn.Module):
    """MITR with InfoNCE estimator - the missing experiment."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg            = cfg
        self.encoder        = AutoModel.from_pretrained(cfg.model_name)
        h                   = self.encoder.config.hidden_size
        self.pre_classifier = nn.Linear(h, h)
        self.classifier     = nn.Linear(h, cfg.num_labels)
        self.dropout        = nn.Dropout(0.1)
        self.mi_estimator   = InfoNCEMI(h, h, cfg.club_hidden)
        self._step          = 0

    def _lam(self):
        if self._step >= self.cfg.mi_warmup_steps:
            return self.cfg.mi_lambda
        return self.cfg.mi_lambda * (self._step / max(1, self.cfg.mi_warmup_steps))

    def forward(self, input_ids, attention_mask, labels=None, is_training=False):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                           output_hidden_states=True)
        cls    = out.last_hidden_state[:, 0]
        cls    = self.dropout(F.relu(self.pre_classifier(cls)))
        logits = self.classifier(cls)
        result = {"logits": logits, "mi_loss": 0.0}

        if labels is not None:
            task_loss = F.cross_entropy(logits, labels)
            lam       = self._lam()
            if is_training and lam > 0.0:
                hs    = out.hidden_states[1:]  # exclude embedding layer (index 0) -> 10 MI terms for 12-layer models
                diffs = []
                for i in range(len(hs) - 1):
                    d = (hs[i+1] - hs[i]).mean(dim=1)
                    d = F.layer_norm(d, (d.size(-1),))
                    diffs.append(d)
                mi_list = [self.mi_estimator(diffs[i], diffs[i+1])
                           for i in range(len(diffs) - 1)]
                if mi_list:
                    mi_mean          = torch.stack(mi_list).mean()
                    result["mi_loss"] = mi_mean.item()
                    result["loss"]    = (1.0 - lam) * task_loss + lam * mi_mean
                else:
                    result["loss"] = task_loss
            else:
                result["loss"] = task_loss
            if is_training:
                self._step += (1.0 / self.cfg.grad_accum_steps)  # grad-accum aware warmup pacing
        return result

print("Models defined.")

In [ ]:
def build_optimizer_scheduler(model, train_loader, cfg):
    no_decay = {"bias", "LayerNorm.weight"}
    grouped  = [
        {"params": [p for n, p in model.named_parameters()
                    if p.requires_grad and not any(nd in n for nd in no_decay)],
         "weight_decay": cfg.weight_decay},
        {"params": [p for n, p in model.named_parameters()
                    if p.requires_grad and any(nd in n for nd in no_decay)],
         "weight_decay": 0.0},
    ]
    try:
        opt = torch.optim.AdamW(grouped, lr=cfg.learning_rate, fused=True)
    except TypeError:
        opt = torch.optim.AdamW(grouped, lr=cfg.learning_rate)
    total_steps  = len(train_loader) * cfg.num_epochs // cfg.grad_accum_steps
    warmup_steps = int(total_steps * cfg.warmup_ratio)
    sched        = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)
    return opt, sched


def train_one_epoch(model, loader, optimizer, scheduler, device, dtype,
                    grad_accum=1, max_grad_norm=1.0, is_mitr=False):
    model.train()
    total_loss = total_mi = n = 0
    use_amp = (dtype != torch.float32)
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(loader, desc="  train", leave=False)):
        ids  = batch["input_ids"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)
        lbls = batch["labels"].to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=dtype, enabled=use_amp):
            out = model(ids, mask, labels=lbls, is_training=is_mitr)
        (out["loss"] / grad_accum).backward()
        if (step + 1) % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total_loss += out["loss"].item()
        mi_val      = out.get("mi_loss", 0.0)
        total_mi   += mi_val if isinstance(mi_val, float) else float(mi_val)
        n          += 1
    return {"train_loss": total_loss / n, "train_mi_loss": total_mi / n}


@torch.no_grad()
def eval_accuracy(model, loader, device, dtype):
    model.eval()
    correct = total = 0; val_loss = 0.0
    use_amp = (dtype != torch.float32)
    for batch in loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)
        with torch.autocast(device_type=device.type, dtype=dtype, enabled=use_amp):
            out = model(ids, mask, labels=lbls)
        preds    = out["logits"].argmax(-1)
        correct += (preds == lbls).sum().item()
        total   += lbls.size(0)
        if "loss" in out: val_loss += out["loss"].item()
    return {"accuracy": correct / total, "val_loss": val_loss / len(loader)}


@torch.no_grad()
def eval_contradiction_rate(model, pair_loader, device, dtype):
    model.eval()
    contradictions = fwd_correct = neg_correct = total = 0
    use_amp = (dtype != torch.float32)
    for batch in pair_loader:
        fwd_ids  = batch["fwd_input_ids"].to(device)
        fwd_mask = batch["fwd_attention_mask"].to(device)
        neg_ids  = batch["neg_input_ids"].to(device)
        neg_mask = batch["neg_attention_mask"].to(device)
        fwd_lbl  = batch["fwd_label"].to(device)
        neg_lbl  = batch["neg_label"].to(device)
        with torch.autocast(device_type=device.type, dtype=dtype, enabled=use_amp):
            fwd_out = model(fwd_ids, fwd_mask)
            neg_out = model(neg_ids, neg_mask)
        fwd_pred = fwd_out["logits"].argmax(-1)
        neg_pred = neg_out["logits"].argmax(-1)
        contradictions += (fwd_pred == neg_pred).sum().item()
        fwd_correct    += (fwd_pred == fwd_lbl).sum().item()
        neg_correct    += (neg_pred == neg_lbl).sum().item()
        total          += fwd_lbl.size(0)
    return {
        "contradiction_rate": contradictions / total,
        "fwd_accuracy":       fwd_correct / total,
        "neg_accuracy":       neg_correct / total,
    }

print("Training utilities defined.")

In [ ]:
def run_experiment(name, model, cfg, is_mitr, train_loader, val_loader, pair_loader):
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    model  = model.to(DEVICE)
    opt, sched = build_optimizer_scheduler(model, train_loader, cfg)
    history    = defaultdict(list)

    for epoch in range(1, cfg.num_epochs + 1):
        tr  = train_one_epoch(model, train_loader, opt, sched, DEVICE, DTYPE,
                               grad_accum=cfg.grad_accum_steps,
                               max_grad_norm=cfg.max_grad_norm,
                               is_mitr=is_mitr)
        val = eval_accuracy(model, val_loader, DEVICE, DTYPE)
        history["epoch"].append(epoch)
        history["train_loss"].append(tr["train_loss"])
        history["train_mi_loss"].append(tr["train_mi_loss"])
        history["val_accuracy"].append(val["accuracy"])
        print(f"  Ep {epoch}/{cfg.num_epochs}  "
              f"train={tr['train_loss']:.4f}  mi={tr['train_mi_loss']:.4f}  "
              f"acc={val['accuracy']:.4f}")

    print("  Evaluating contradiction rate...")
    contra = eval_contradiction_rate(model, pair_loader, DEVICE, DTYPE)
    print(f"  Contradiction rate : {contra['contradiction_rate']:.4f}")

    del model, opt, sched
    torch.cuda.empty_cache()

    return {
        "name":               name,
        "history":            dict(history),
        "final_accuracy":     history["val_accuracy"][-1],
        "contradiction_rate": contra["contradiction_rate"],
    }

print("Runner defined. Starting experiments -- this will take ~30-50 min on T4.")

In [ ]:
# -- RUN ALL EXPERIMENTS --
# 2 backbones x 2 configs (baseline + InfoNCE) = 4 runs total

all_results = {}
_dl_kw = dict(pin_memory=(DEVICE.type == "cuda"), num_workers=2, persistent_workers=True)

for backbone in BACKBONES:
    print(f"\n{'#'*60}")
    print(f"#  BACKBONE: {backbone}")
    print(f"{'#'*60}")

    tok = AutoTokenizer.from_pretrained(backbone)

    train_ds   = LogicDataset(train_raw, tok, 256)
    val_ds     = LogicDataset(val_raw,   tok, 256)
    pair_ds    = ContradictionPairDataset(contradiction_pairs, tok, 256)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  **_dl_kw)
    val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, **_dl_kw)
    pair_loader  = DataLoader(pair_ds,  batch_size=32, shuffle=False, **_dl_kw)

    short = backbone.split("/")[-1]

    # Baseline
    set_seed(42)
    cfg   = Config(model_name=backbone)
    model = BaselineClassifier(backbone)
    all_results[f"{short}_baseline"] = run_experiment(
        f"Baseline {short}", model, cfg, is_mitr=False,
        train_loader=train_loader, val_loader=val_loader, pair_loader=pair_loader,
    )

    # MITR-InfoNCE
    set_seed(42)
    cfg   = Config(model_name=backbone, mi_strategy="infonce")
    model = MITRInfoNCEClassifier(cfg)
    all_results[f"{short}_infonce"] = run_experiment(
        f"MITR-InfoNCE {short}", model, cfg, is_mitr=True,
        train_loader=train_loader, val_loader=val_loader, pair_loader=pair_loader,
    )

with open("experiment_results/infonce_results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=float)

print("\nAll experiments complete.")

In [ ]:
# -- Results table + charts (6-panel layout matching original notebooks) --

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")

COLORS  = {"baseline": "#78909C", "infonce": "#9C27B0"}
MARKERS = {"baseline": "o", "infonce": "D"}

fig = plt.figure(figsize=(24, 16))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── 1. BERT validation accuracy curve ────────────────────────
ax = fig.add_subplot(gs[0, 0])
short = "bert-base-uncased"
for strategy in ["baseline", "infonce"]:
    r = all_results[f"{short}_{strategy}"]
    ax.plot(r["history"]["epoch"], r["history"]["val_accuracy"],
            color=COLORS[strategy], marker=MARKERS[strategy],
            label=strategy.capitalize(), linewidth=2)
ax.set(title="bert-base-uncased\nValidation Accuracy", xlabel="Epoch", ylabel="Accuracy")
ax.legend(fontsize=9)

# ── 2. RoBERTa validation accuracy curve ─────────────────────
ax = fig.add_subplot(gs[0, 1])
short = "roberta-base"
for strategy in ["baseline", "infonce"]:
    r = all_results[f"{short}_{strategy}"]
    ax.plot(r["history"]["epoch"], r["history"]["val_accuracy"],
            color=COLORS[strategy], marker=MARKERS[strategy],
            label=strategy.capitalize(), linewidth=2)
ax.set(title="roberta-base\nValidation Accuracy", xlabel="Epoch", ylabel="Accuracy")
ax.legend(fontsize=9)

# ── 3. MI Regularisation Loss (InfoNCE only, stability check) ─
ax = fig.add_subplot(gs[0, 2])
for backbone in BACKBONES:
    short = backbone.split("/")[-1]
    r     = all_results[f"{short}_infonce"]
    ax.plot(r["history"]["epoch"], r["history"]["train_mi_loss"],
            marker="D", linewidth=2, label=f"InfoNCE {short[:4]}")
ax.set(title="MI Regularisation Loss\n(stability: should not explode)",
       xlabel="Epoch", ylabel="MI Loss")
ax.legend(fontsize=9)

# ── 4. Cross-model final accuracy bar chart ───────────────────
ax    = fig.add_subplot(gs[1, 0])
x     = np.arange(len(BACKBONES))
width = 0.35
for i, strategy in enumerate(["baseline", "infonce"]):
    vals = [all_results[f"{b.split('/')[-1]}_{strategy}"]["final_accuracy"]
            for b in BACKBONES]
    bars = ax.bar(x + i*width, vals, width, label=strategy.capitalize(),
                  color=COLORS[strategy], edgecolor="black")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                f"{v:.3f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
ax.set_xticks(x + width/2)
ax.set_xticklabels([b.split("/")[-1] for b in BACKBONES], fontsize=9)
ax.set(title="Final Accuracy Comparison", ylabel="Accuracy")
ax.legend(fontsize=9)

# ── 5. Contradiction rate per backbone ───────────────────────
ax = fig.add_subplot(gs[1, 1])
for i, strategy in enumerate(["baseline", "infonce"]):
    vals = [all_results[f"{b.split('/')[-1]}_{strategy}"]["contradiction_rate"]
            for b in BACKBONES]
    bars = ax.bar(x + i*width, vals, width, label=strategy.capitalize(),
                  color=COLORS[strategy], edgecolor="black")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                f"{v:.4f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
ax.set_xticks(x + width/2)
ax.set_xticklabels([b.split("/")[-1] for b in BACKBONES], fontsize=9)
ax.set(title="Contradiction Rate (lower = better)", ylabel="Fraction contradictory")
ax.legend(fontsize=9)

# ── 6. Summary table ─────────────────────────────────────────
ax = fig.add_subplot(gs[1, 2]); ax.axis("off")
rows = []
for backbone in BACKBONES:
    short     = backbone.split("/")[-1]
    bl        = all_results[f"{short}_baseline"]
    mitr      = all_results[f"{short}_infonce"]
    acc_d     = (mitr["final_accuracy"]   - bl["final_accuracy"])   * 100
    contra_d  = (bl["contradiction_rate"] - mitr["contradiction_rate"]) * 100
    rows.append([short[:8]+" Baseline", f"{bl['final_accuracy']:.4f}",
                 "--", f"{bl['contradiction_rate']:.4f}", "--"])
    rows.append([short[:8]+" InfoNCE",  f"{mitr['final_accuracy']:.4f}",
                 f"{acc_d:+.2f}%", f"{mitr['contradiction_rate']:.4f}",
                 f"{contra_d:+.2f}%"])

tbl = ax.table(
    cellText=rows,
    colLabels=["Model", "Acc", "Acc Delta", "Contra", "Contra Red."],
    cellLoc="center", loc="center", bbox=[0, 0.0, 1, 0.95],
)
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor("#37474F")
        cell.set_text_props(color="white", fontweight="bold")
    elif row % 2 == 0:
        cell.set_facecolor("#F5F5F5")
ax.set_title("Summary", fontweight="bold", pad=15)

fig.suptitle("MITR-InfoNCE Generalization: BERT & RoBERTa\n"
             "Filling the gap from roberta_bert_results.md",
             fontsize=15, fontweight="bold", y=1.01)

plt.savefig("experiment_results/infonce_generalization.pdf", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# -- Summary table --

print("\n" + "="*70)
print("RESULTS SUMMARY: MITR-InfoNCE on BERT & RoBERTa")
print("="*70)
print(f"{'Model':<30} {'Accuracy':>10} {'Acc Delta':>10} {'Contra Rate':>12} {'Contra Delta':>13}")
print("-"*70)

for backbone in BACKBONES:
    short  = backbone.split("/")[-1]
    bl     = all_results[f"{short}_baseline"]
    mitr   = all_results[f"{short}_infonce"]

    acc_d  = (mitr["final_accuracy"]     - bl["final_accuracy"])     * 100
    con_d  = (bl["contradiction_rate"]   - mitr["contradiction_rate"]) * 100

    print(f"{short + ' Baseline':<30} {bl['final_accuracy']:>10.4f} {'--':>10} "
          f"{bl['contradiction_rate']:>12.4f} {'--':>13}")
    print(f"{short + ' MITR-InfoNCE':<30} {mitr['final_accuracy']:>10.4f} "
          f"{acc_d:>+9.2f}% {mitr['contradiction_rate']:>12.4f} {con_d:>+12.2f}%")
    print()

# Reference DistilBERT result for comparison
print("Reference (DistilBERT, from results.md, 5 epochs, A100):")
print(f"  DistilBERT Baseline : acc=0.6980  contra=0.4300")
print(f"  DistilBERT InfoNCE  : acc=0.7000 (+0.20%)  contra=0.4120 (-1.80%)")

## Discussion

MITR-InfoNCE partially supports the hypothesis that the contrastive estimator generalizes to 12-layer models. On BERT, InfoNCE improved both accuracy (0.7053 vs 0.7027, +0.26%) and contradiction rate (0.4220 vs 0.4800, -5.80%), replicating the DistilBERT result that InfoNCE is the only strategy to improve both metrics simultaneously. On RoBERTa, InfoNCE matched baseline accuracy (0.7373 vs 0.7413, -0.40%) but worsened contradiction rate (0.4400 vs 0.4020, +3.80%).

This is a direct reversal of the Cosine/CKA pattern in `roberta_bert_results.md`, where those strategies hurt BERT but helped RoBERTa. InfoNCE flips that entirely. The backbone-dependent effect is real but interacts with estimator choice in a way the original experiments could not detect by excluding InfoNCE. No single strategy dominates across both backbones, strengthening the paper's core claim that MITR's effect depends jointly on pretraining-induced layer structure and MI estimation method.